
# <div align="center"> Stanford RNA 3D Folding 
    

<div align="center"> <img src="https://www.kaggle.com/competitions/87793/images/header" width="300"></div><br>

**From the competition material**: For each sequence in the test set, you can predict <ins>five structures</ins>. Your notebook should look for a file test_sequences.csv and output submission.csv. This file should contain x, y, z coordinates of the C1' atom in each residue across your predicted structures 1 to 5:


**ID**,**resname**,**resid**,**x_1**,**y_1**,**z_1**,... **x_5**,**y_5**,**z_5** <br>
R1107_1,G,1,-7.561,9.392,9.361,... -7.301,9.023,8.932<br>
R1107_2,G,1,-8.02,11.014,14.606,... -7.953,10.02,12.127<br>
etc.<br>


**Evaluation**: Submissions are scored using <ins>TM-score</ins> ("template modeling" score), which goes from 0.0 to 1.0 (higher is better):


# 📚 Libraries

In [ ]:
import pandas as pd
# Import the required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# 🔍 Exploratory Data Analysis (EDA)

In [ ]:
# Read training sequences
train_sequences = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/train_sequences.csv')
# Read training labels
train_labels = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/train_labels.csv')
# Read validation sequences
validation_sequences = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/validation_sequences.csv')
# Read validation labels
validation_labels = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/validation_labels.csv')

In [ ]:
train_sequences.shape

### train_sequences.csv

In [ ]:
# Print the training sequence row and column counts
print(f'train_sequences has {train_sequences.shape[0]} rows and {train_sequences.shape[1]} columns')
# Print the total number of missing values in the training sequences
print(f'train_sequences has {train_sequences.isna().sum().sum()} NAs.')
# Preview the first 3 rows
train_sequences.head(12)

# sequence column...specifically the length of the sequences.

In [ ]:
# Set the figure size
plt.figure(figsize=(15, 8))
# Calculate sequence lengths and create a new DataFrame
train_sequences['length'] = train_sequences['sequence'].str.len()
# Draw a boxplot
sns.boxplot(x='length', data=train_sequences)
plt.xlabel("Sequence length")
plt.ylabel("Sequence")
plt.title("Boxplot of sequence length values (with outliers)")
plt.grid(True)
plt.show()

In [ ]:
# Set the figure size
plt.figure(figsize=(15, 8))

# Calculate sequence lengths and create a new DataFrame
train_sequences['length'] = train_sequences['sequence'].str.len()

# Draw a boxplot without outliers
sns.boxplot(x='length', data=train_sequences, showfliers=False)
plt.xlabel("Sequence length")
plt.ylabel("Sequence")
plt.title("Boxplot of sequence length values (without outliers)")
plt.grid(True)
plt.show()

### Examine temporary_cutoff to see how published sequence lengths have changed over time

In [ ]:
# Create the 'cutoff_year' and 'sequence_length' columns
train_sequences['cutoff_year'] = pd.to_datetime(train_sequences['temporal_cutoff']).dt.year
train_sequences['sequence_length'] = train_sequences['sequence'].str.len()

# Calculate the average sequence length for each year
avg_sequence_length = (
    train_sequences
    .groupby('cutoff_year')
    .agg(avg_sequence_length=('sequence_length', 'mean'))
    .reset_index()
)

# Set the figure size
plt.figure(figsize=(15, 8))

# Plot observed sequence lengths and a polynomial smooth
sns.regplot(x='cutoff_year', y='avg_sequence_length', data=avg_sequence_length, 
            order=3, ci=None, label='Smoothed Line', color='blue')
plt.plot(avg_sequence_length['cutoff_year'], avg_sequence_length['avg_sequence_length'], 
         color='black', label='Actual Sequence Length')

# Add axis labels and a title
plt.xlabel("Temporal Cutoff Year")
plt.ylabel("Average Sequence Length discovered")
plt.title("Average length of published sequence over time")
plt.legend()
plt.grid(True)
plt.show()

### Focussing on the target_id...from the competition material:
>"In train_sequences.csv, this is formatted as pdb_id_chain_id, where pdb_id is the id of the entry in the Protein Data Bank and chain_id is the chain id of the monomer in the pdb file
>### Focus on target_id from the competition material...
>"In train_sequences.csv, the format is pdb_id_chain_id, where pdb_id identifies the Protein Data Bank entry and chain_id identifies the monomer chain in the PDB file.

#### Let's first look at how many unique pdb_id's and chain_id's there are in the train_sequences

In [ ]:
# Import the required libraries
import pandas as pd

# Assume train_sequences contains a 'target_id' column
# Extract pdb_id and chain_id
train_sequences[['pdb_id', 'chain_id']] = train_sequences['target_id'].str.split('_', expand=True)

# Count unique pdb_id values
pdb_id_count = train_sequences['pdb_id'].nunique()

# Count unique chain_id values
chain_id_count = train_sequences['chain_id'].nunique()

# Print the results
print(f"There are {pdb_id_count} unique pdb_id's in the dataset.")
print(f"There are {chain_id_count} unique chain_id's in the dataset.")

### Now let's look at which pdb_id's are repeated in the train_sequences

In [ ]:
# Import the required libraries
import pandas as pd

# Assume train_sequences contains a 'target_id' column
# Extract pdb_id and chain_id
train_sequences[['pdb_id', 'chain_id']] = train_sequences['target_id'].str.split('_', expand=True)

# Count occurrences of each pdb_id and select repeated IDs
repeated_pdb_ids = (
    train_sequences
    .groupby('pdb_id')
    .size()
    .reset_index(name='total')
    .query('total > 1')
    .sort_values(by='total', ascending=False)
)

# Get the 10 most frequently repeated pdb_id values
top_repeated_pdb_ids = repeated_pdb_ids.head(10)

# Print the results
print("In total, there are", repeated_pdb_ids.shape[0], "pdb_id's that are repeated in train_sequences.")
print("Top 10 most repeated pdb_id's:")
top_repeated_pdb_ids# In total, there are 60 pdb_id's that are repeated in train_sequences, with the top 10 most repeated shown below

In [ ]:
train_sequences[train_sequences['pdb_id']=='4V5Z']

### train_labels.csv

In [ ]:
# Assume train_labels is a Pandas DataFrame
# Print the row and column counts
print(f'train_labels has {train_labels.shape[0]} rows and {train_labels.shape[1]} columns.')
# Print the total number of missing values
print(f'train_labels has {train_labels.isna().sum().sum()} NAs.')
# Preview the data
train_labels.head(3)

In [ ]:
train_labels.shape

In [ ]:
# Find rows with missing values and display the first 3
na_rows = train_labels[train_labels.isna().any(axis=1)]
# Print the first 3 rows
na_rows.head(3)

In [ ]:
na_rows.shape

### Let's create a boxplot of the coordinate values for x, y, & z 

In [ ]:
# Select x_1, y_1, and z_1 and reshape to long format
train_labels_long = train_labels[['x_1', 'y_1', 'z_1']].melt(var_name='coordinate', value_name='value')
# Set the figure size
plt.figure(figsize=(15, 8))
# Draw a boxplot
sns.boxplot(x='value', y='coordinate', data=train_labels_long, hue='coordinate', palette='YlGnBu', dodge=False)
# Add axis labels and a title
plt.xlabel("Coordinate Values")
plt.ylabel("Coordinates")
plt.title("Boxplot of coordinate values in train_labels")
plt.legend(title='Coordinate', loc='upper right')
plt.grid(True)
plt.show()

### Resname occurrences in train_labels

In [ ]:
# Count occurrences of each resname
resname_counts = train_labels.groupby('resname').size().reset_index(name='total')
# Set the figure size
plt.figure(figsize=(15, 8))
# Draw a bar chart
sns.barplot(x='resname', y='total', data=resname_counts, palette='Blues')
# Add count labels to the bars
for index, row in resname_counts.iterrows():
    plt.text(index, row['total'], row['total'], ha='center', va='bottom')
# Set axis labels and a title
plt.xlabel("Resname")
plt.ylabel("Count")
plt.title("Barplot of resname occurrences in train_labels")
# Hide the legend
plt.legend().set_visible(False)
# Use a minimal theme
sns.set_theme(style="whitegrid")
# Display the figure
plt.show()

### validation_sequences.csv

In [ ]:
# Import the required libraries
import pandas as pd

# Assume validation_sequences is a Pandas DataFrame
# Print the row and column counts
print(f'validation_sequences has {validation_sequences.shape[0]} rows and {validation_sequences.shape[1]} columns.')

# Print the total number of missing values
print(f'validation_sequences has {validation_sequences.isna().sum().sum()} NAs.')

# Preview the data
validation_sequences.head(3)

### Let's recreate the boxplot of the sequence lengths we used in train_sequences, using outliers = FALSE

In [ ]:
# Import the required libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assume validation_sequences is a Pandas DataFrame
# Calculate sequence lengths and create a new column
validation_sequences['length'] = validation_sequences['sequence'].str.len()

# Set the figure size
plt.figure(figsize=(15, 8))

# Draw a boxplot
sns.boxplot(x='length', data=validation_sequences, showfliers=False)

# Set axis labels and a title
plt.xlabel("Sequence length")
plt.ylabel("Sequence")
plt.title("Boxplot of sequence length values (without outliers)")

# Use a minimal theme
sns.set_theme(style="whitegrid")

# Display the figure
plt.show()

In [ ]:
# Import the required libraries
import pandas as pd

# Assume train_sequences and validation_sequences are Pandas DataFrames
# Calculate the median sequence length in train_sequences
train_sequence_med = train_sequences['sequence'].str.len().median()

# Calculate the median sequence length in validation_sequences
valid_sequence_med = validation_sequences['sequence'].str.len().median()

# Print the results
print(f'The train_sequences median sequence length is {train_sequence_med}.')
print(f'The validation_sequences median sequence length is {valid_sequence_med}.')

### validation_labels.csv

In [ ]:
# Import the required libraries
import pandas as pd

# Assume validation_labels is a Pandas DataFrame
# Print the row and column counts
print(f'validation_labels has {validation_labels.shape[0]} rows and {validation_labels.shape[1]} columns.')

# Print the total number of missing values
print(f'validation_labels has {validation_labels.isna().sum().sum()} NAs.')

print("Note validation_labels has 123 columns; train_labels has 6.")
# Preview the data
validation_labels.head(3)

### The number of NAs is misleading; from a discussion post [here](https://www.kaggle.com/competitions/stanford-rna-3d-folding/discussion/565746), the -1e+18 should be interpreted as NA/NaN values.

### We will recreate the resname boxplot from train_labels from earlier:

In [ ]:
# Import the required libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assume validation_labels is a Pandas DataFrame
# Count occurrences of each resname
resname_counts = validation_labels.groupby('resname').size().reset_index(name='total')

# Set the figure size
plt.figure(figsize=(15, 8))

# Draw a bar chart
sns.barplot(x='resname', y='total', data=resname_counts, palette='Blues', order=resname_counts.sort_values('total')['resname'])

# Add count labels to the bars
for index, row in resname_counts.iterrows():
    plt.text(index, row['total'], row['total'], ha='center', va='bottom')

# Set axis labels and a title
plt.xlabel("Resname")
plt.ylabel("Count")
plt.title("Barplot of resname occurrences in validation_labels")

# Hide the legend
plt.legend().set_visible(False)

# Use a minimal theme
sns.set_theme(style="whitegrid")

# Display the figure
plt.show()

# Explanation
print("Note there are no missing or 'X' resnames.")
print("Also of note is in this dataset, 'U' occurs more frequently than 'A'. 'C' and 'G' maintain their ordering.")

### Submission format

In [ ]:
# Import the required libraries
import pandas as pd

# Read the sample submission
sample_submission = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/sample_submission.csv')

# Print the row and column counts
print(f'There are {sample_submission.shape[0]} rows and {sample_submission.shape[1]} columns.')

# Preview the first 6 rows
print("Note: It is required to submit 5 sets of coordinate plots (x,y,z), per ID, resname & resid from the target_id, which is 18 columns.")
print("For rows in the submission, we need one row for each resname/resid for each target.")
print("From the sample submission, we need 2515 rows.")
sample_submission.head(6)

# Model training


### Generate an output file from test_sequence

In [ ]:
# Import the required libraries
import pandas as pd

# Read the test sequences
test_sequences = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/test_sequences.csv')

# Print the row and column counts
print(f'There are {test_sequences.shape[0]} rows and {test_sequences.shape[1]} columns.')

# Explanation
print("Reading in the test sequences, we can see we have 12 rows and the same 5 columns found in train & validation sequences.")
test_sequences.head()

In [ ]:
# Import the required libraries
import pandas as pd

# Read the test sequences
test_sequences = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/test_sequences.csv')

# Calculate the total length of all sequences
total_length = test_sequences['sequence'].str.len().sum()

print(f'Total length of sequences in test_sequences is {total_length}.')

# Create a DataFrame to hold submission data
submission_df = pd.DataFrame(columns=['target_id', 'resname', 'resid', 'x', 'y', 'z'])  # Define columns as needed

# Explanation
print("Now, we need to create the df to house our submission.")

In [ ]:
# This function will accept a test_sequence ID & sequence and return df in format of submission (one resid/resname per row in sequence)
parse_target <- function(tmp_ID, tmp_sequence){
    seq_length <- nchar(tmp_sequence)
    tmp_df=data.frame(matrix(ncol = 3, nrow = seq_length)) 
    colnames(tmp_df) <- c('ID','resname','resid')
    tmp_df$resname <- unlist(strsplit(tmp_sequence,split=''))
    tmp_df$ID <- tmp_ID
    tmp_df$resid <- seq(1:seq_length)

    return(tmp_df)
}
# Create the test_clean df in the format of the submission
test_id_seq <- test_sequences %>% select(target_id, sequence)
test_clean <- data.frame(matrix(ncol = 3, nrow = 0)) 
colnames(test_clean) <- c('ID','resname','resid')

# For each target_id / sequence, apply the function and rbind to previous results
for(i in 1:nrow(test_id_seq)){
    tmp_df <- as.data.frame(mapply(parse_target,test_id_seq$target_id[i],test_id_seq$sequence[i], SIMPLIFY=FALSE))
    colnames(tmp_df) <- c('ID','resname','resid')
    test_clean <- rbind(test_clean,tmp_df)
}

print(paste('There are',nrow(test_clean),'rows in the test_clean df.'))
print(head(test_clean))

# That works...now let's get back to the training data, add some features, & train a model.

# 🎯 Model Training

In [ ]:
# Function to create some features based off resname combinations & sequence length
feat_eng <- function(df){
    df <- df %>% mutate(seq_length = nchar(sequence), 
               A_cnt = str_count(sequence,"A"), 
               C_cnt = str_count(sequence,"C"), 
               U_cnt = str_count(sequence,"U"), 
               G_cnt = str_count(sequence,"G"),                  
               AC_cnt = str_count(sequence,"AC"), 
               AU_cnt = str_count(sequence,"AU"), 
               AG_cnt = str_count(sequence,"AG"),
               CA_cnt = str_count(sequence,"CA"), 
               CU_cnt = str_count(sequence,"CU"), 
               CG_cnt = str_count(sequence,"CG"),
               UA_cnt = str_count(sequence,"UA"), 
               UC_cnt = str_count(sequence,"UC"), 
               UG_cnt = str_count(sequence,"UG"),
               GA_cnt = str_count(sequence,"GA"), 
               GC_cnt = str_count(sequence,"GC"), 
               GU_cnt = str_count(sequence,"GU"), 
               AA_cnt = str_count(sequence,"AA"), 
               CC_cnt = str_count(sequence,"CC"), 
               UU_cnt = str_count(sequence,"UU"), 
               GG_cnt = str_count(sequence,"GG"),
               begin_seq = substr(sequence,1,1),
               end_seq = substr(sequence,nchar(sequence),nchar(sequence))) %>% select(-c(sequence, temporal_cutoff, description, all_sequences))
    return(df)
}
train_sequences <- feat_eng(train_sequences)

# Create the target_id from ID to facilitate joining with the meta_train data we just created
train_labels <- train_labels %>% group_by(ID) %>% mutate(target_id = paste(unlist(strsplit(ID,'_'))[1],unlist(strsplit(ID,'_'))[2],sep='_'))

# Create the final df to train
train_data_clean <- merge(train_labels, train_sequences, by="target_id", all.x = TRUE)

# For now, we will impute the missing x_1,y_1,z_1 values with the group average for that target_id, resname
train_data_clean <- train_data_clean %>% group_by(target_id, resname) %>% mutate(x_1 = ifelse(is.na(x_1),mean(x_1, na.rm=TRUE),x_1), y_1 = ifelse(is.na(y_1),mean(y_1, na.rm=TRUE),y_1), z_1 = ifelse(is.na(z_1),mean(z_1, na.rm=TRUE),z_1)) %>% ungroup() %>% select(-target_id)

# The above imputation for train_data_clean doesn't take care of all NAs...so for now we will remove them(~ 1979 rows)
train_data_clean <- train_data_clean %>% na.exclude() 
print(paste('There are ',sum(is.na(train_data_clean)),"NA's in the dataset!"))

In [ ]:
features <- train_data_clean %>% select(-c(x_1,y_1,z_1, ID)) %>% colnames()  

### For now, our model approach will consist of 3 separate models for x,y,z values

In [ ]:
# Train model for X

train_hframe = train_data_clean %>%  select(-c(ID,y_1,z_1)) %>% as.h2o() # will need to separate out x_1,y_1,z_1 & ID for each iteration
train_hframe$resname <- as.factor(train_hframe$resname)
train_hframe$begin_seq <- as.factor(train_hframe$begin_seq)
train_hframe$end_seq <- as.factor(train_hframe$end_seq)
# split the data into training and validation sets:
splits <- h2o.splitFrame(data = train_hframe, ratio = 0.8, seed = 1)
train <- splits[[1]]
valid <- splits[[2]]
xgb_x <- h2o.xgboost(x = features,
        y = 'x_1',
        nfolds = 5,
        seed = 1,
        booster = 'gbtree',
        ntrees=200,
        max_depth=15,
        keep_cross_validation_predictions = TRUE,
        training_frame = train_hframe,
        validation_frame = valid)

print(h2o.mse(xgb_x,train=TRUE,valid=TRUE))

# Train model for Y
train_hframe = train_data_clean %>% select(-c(ID,x_1,z_1)) %>% as.h2o() # will need to separate out x_1,y_1,z_1 & ID for each iteration
train_hframe$resname <- as.factor(train_hframe$resname)
train_hframe$begin_seq <- as.factor(train_hframe$begin_seq)
train_hframe$end_seq <- as.factor(train_hframe$end_seq)
# split the data into training and validation sets:
splits <- h2o.splitFrame(data = train_hframe, ratio = 0.8, seed = 1)
train <- splits[[1]]
valid <- splits[[2]]
xgb_y <- h2o.xgboost(x = features,
        y = 'y_1',
        nfolds = 5,
        seed = 1,
        booster = 'gbtree',
        ntrees=200,
        max_depth=15,
        keep_cross_validation_predictions = TRUE,
        training_frame = train_hframe,
        validation_frame = valid)

print(h2o.mse(xgb_y,train=TRUE,valid=TRUE))

# Train model for Z
train_hframe = train_data_clean %>% select(-c(ID,y_1,x_1)) %>% as.h2o() # will need to separate out x_1,y_1,z_1 & ID for each iteration
train_hframe$resname <- as.factor(train_hframe$resname)
train_hframe$begin_seq <- as.factor(train_hframe$begin_seq)
train_hframe$end_seq <- as.factor(train_hframe$end_seq)
# split the data into training and validation sets:
splits <- h2o.splitFrame(data = train_hframe, ratio = 0.8, seed = 1)
train <- splits[[1]]
valid <- splits[[2]]
xgb_z <- h2o.xgboost(x = features,
        y = 'z_1',
        nfolds = 5,
        seed = 1,
        booster = 'gbtree',
        ntrees=200,
        max_depth=15,
        keep_cross_validation_predictions = TRUE,
        training_frame = train_hframe,
        validation_frame = valid)

print(h2o.mse(xgb_z,train=TRUE,valid=TRUE))

In [ ]:
# Create features on test sequence
test_sequences <- feat_eng(test_sequences)

# Merge test_clean we created earlier with test_sequences
test_data_clean <- merge(test_clean, test_sequences, by.x="ID",by.y="target_id", all.x = TRUE)
test_data_clean$resname <- as.factor(test_data_clean$resname)
test_data_clean$begin_seq <- as.factor(test_data_clean$begin_seq)
test_data_clean$end_seq <- as.factor(test_data_clean$end_seq)

In [ ]:
preds_x = h2o.predict(xgb_x,newdata = as.h2o(test_data_clean %>% select(-ID)))
preds_y = h2o.predict(xgb_y,newdata = as.h2o(test_data_clean %>% select(-ID)))
preds_z = h2o.predict(xgb_z,newdata = as.h2o(test_data_clean %>% select(-ID)))

# 🤞 Submission

### For now, we will use the same predictions for all x,y,z combinations.  Future version will incorporate different methods for these predictions.

In [ ]:
submission <- data.frame(test_data_clean$ID, 
                         test_data_clean$resname, 
                         test_data_clean$resid) 

submission <- cbind(submission, 
                         as.data.frame(preds_x),
                         as.data.frame(preds_y),
                         as.data.frame(preds_z),
                    
                         as.data.frame(preds_x),
                         as.data.frame(preds_y),
                         as.data.frame(preds_z),
                    
                         as.data.frame(preds_x),
                         as.data.frame(preds_y),
                         as.data.frame(preds_z),
                    
                         as.data.frame(preds_x),
                         as.data.frame(preds_y),
                         as.data.frame(preds_z),
                    
                         as.data.frame(preds_x),
                         as.data.frame(preds_y),
                         as.data.frame(preds_z))
colnames(submission) <- sample_submission %>% colnames()

### My first two submissions in R errored at the scoring phase.  This was due to the ID columns not being in the same order as the sample_submission, so we will create a simple sort column based off the sample_submission, and apply that to our submission df.

In [ ]:
# Update ID to match submission ID column
submission <- submission %>% mutate(ID = paste(ID,resid,sep="_"))

# Create a sort_order column within the sample_submission so we can apply it to our submission
sample_submission$sort_order <- seq(1:nrow(sample_submission))
submission <- merge(submission,sample_submission %>% select(ID,sort_order), by='ID',all.x=TRUE) %>% arrange(sort_order) %>% select(-sort_order)

In [ ]:
submission %>% head(3)

In [ ]:
write_csv(submission, 'submission.csv')